# Stuttering Detection — Research Demonstration Notebook

This notebook is a **cleaned, narrated demonstration** of the
stuttering-detection pipeline. The reusable implementation lives in
`src/` and `config/`; this notebook imports and calls it rather than
duplicating logic, so behavior here always matches the scripts in
`scripts/`.

No personal paths, machine usernames, or sensitive outputs from the
original exploratory notebook are included here — see
`SECURITY_AUDIT.md` for details. Configure your dataset location via
`.env` (copied from `.env.example`) or the `DATASET_ROOT` environment
variable before running the cells below.


In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running this notebook from
# notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from config import config
print("Dataset root:", config.RAW_DATA_DIR)
print("Feature dir :", config.FEATURE_DIR)
print("Model dir   :", config.MODEL_DIR)


## 1. Dataset Loading

Loads `audio_dataset.csv` from the configured dataset root and reports
the top-level composition (normal vs. stuttering source recordings).
See `data/README.md` for the expected dataset layout.


In [ ]:
from src.data.load_dataset import load_raw_metadata, summarize_raw_dataset

metadata_csv = config.RAW_DATA_DIR / "audio_dataset.csv"
df = load_raw_metadata(metadata_csv)
summarize_raw_dataset(df)


## 2. Audio Preprocessing

Each recording is resampled to 16 kHz mono, cleaned of non-finite
samples, silence-trimmed (`top_db=30`), filtered by minimum duration
(0.20 s), peak-amplitude normalized, and persisted as a new WAV file.
This step can take a while for the full dataset — cached results are
reused on subsequent runs via `scripts/prepare_dataset.py`.


In [ ]:
from src.data.preprocess import preprocess_dataset

processed_df = preprocess_dataset(
    metadata_csv=metadata_csv,
    processed_audio_dir=config.PROCESSED_AUDIO_DIR,
    output_csv=config.PROCESSED_METADATA_CSV,
    failed_csv=config.FAILED_FILES_CSV,
)

print("Successfully processed:", len(processed_df))


## 3. Label Construction & Speaker-ID Recovery

The three original recording categories (`normal`, `fluent`,
`dysfluent`) are consolidated into the binary classification target:
`normal, fluent -> non_stuttered (0)`; `dysfluent -> stuttered (1)`.
Speaker identifiers are recovered directly (normal-speech set) or
parsed from a `[MF]_####` filename prefix (stuttering set).


In [ ]:
from src.data.load_dataset import build_final_dataset

final_df = build_final_dataset(config.PROCESSED_METADATA_CSV)
final_df.to_csv(config.FINAL_METADATA_FIXED_CSV, index=False)

print("Total recordings:", len(final_df))
print("Unique speakers :", final_df["speaker_id"].nunique())
final_df["binary_label_name"].value_counts()


## 4. Speaker-Independent Dataset Splitting

Partitioning is performed at the **speaker level**, independently
within the normal-speaker and stuttering-speaker groups, then
recombined (70% train / 15% validation / 15% test). A programmatic
check confirms zero speaker overlap between splits.


In [ ]:
import sys as _sys
_sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from prepare_dataset import split_speakers_independently

train_df, val_df, test_df = split_speakers_independently(final_df)

train_df.to_csv(config.TRAIN_CSV, index=False)
val_df.to_csv(config.VALIDATION_CSV, index=False)
test_df.to_csv(config.TEST_CSV, index=False)

for name, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name, "-> recordings:", len(d), "| speakers:", d["speaker_id"].nunique())


## 5. Feature Extraction

Each cleaned waveform is fixed to a 5.0 s window, converted to 40
MFCCs (`n_fft=512`, `hop_length=256`) plus first- and second-order
derivatives, stacked into a `(120, 313)` tensor, and standardized
per-recording/per-coefficient.


In [ ]:
from src.features.feature_extraction import extract_features_for_split

X_train, y_train = extract_features_for_split(
    config.TRAIN_CSV, config.FEATURE_DIR, "X_train.npy", "y_train.npy", "train_failed.csv"
)
X_val, y_val = extract_features_for_split(
    config.VALIDATION_CSV, config.FEATURE_DIR, "X_validation.npy", "y_validation.npy", "validation_failed.csv"
)
X_test, y_test = extract_features_for_split(
    config.TEST_CSV, config.FEATURE_DIR, "X_test.npy", "y_test.npy", "test_failed.csv"
)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)


## 6. Model Architecture

Two CNN architectures are implemented in `src/models/model.py`:
**CNN V1** (32/64/128 filters) and **CNN V2** (16/32/64 filters,
L2-regularized, SpatialDropout). CNN V2 is the architecture referenced
in the accompanying research paper and is used for training below;
pass `version="v1"` to use the earlier architecture instead. See
`docs/methodology.md` for the full layer-by-layer specification and
training hyperparameters for both.


In [ ]:
from src.models.model import build_model

model = build_model("v2", input_shape=(X_train.shape[1], X_train.shape[2], 1))
model.summary()


## 7. Training

Trains the selected CNN with the exact optimizer, loss, callbacks, and
hyperparameters implemented in the original notebook for that
architecture (see `docs/methodology.md` §8). Random seeds are fixed for
reproducibility.


In [ ]:
from src.training.train import train_model

model, history, checkpoint_path, final_model_path = train_model(
    version="v2",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    model_dir=config.MODEL_DIR,
)

print("Best checkpoint:", checkpoint_path)
print("Final model    :", final_model_path)


## 8. Evaluation

Computes accuracy, precision, recall, F1, and ROC-AUC on the held-out,
speaker-disjoint test set, and saves a confusion matrix and training
curves under `results/`.


In [ ]:
from src.evaluation.evaluate import evaluate_model

metrics = evaluate_model(
    model=model,
    X_test=X_test,
    y_test=y_test,
    results_dir=config.RESULTS_DIR,
    prefix="cnn_v2",
    history=history,
)

metrics


## 9. Results

The reported test-set performance for CNN V2 (as printed by the
original notebook and reproduced in `README.md` /
`docs/methodology.md`) was:

| Metric | Value |
|---|---:|
| Accuracy | 91.54% |
| Precision (Stuttered) | 71.77% |
| Recall (Stuttered) | 96.35% |
| F1-score (Stuttered) | 82.26% |
| ROC-AUC | 96.89% |

Re-running the cells above on your own copy of the dataset should
reproduce comparable numbers (subject to the reproducibility notes in
`README.md`), though exact figures may vary slightly depending on
hardware/library versions, which were not recorded in the original
implementation.

See `docs/methodology.md` for the full methodology, known
limitations, and recommended future work.
